In [ ]:
# Base di calcolo presenze di combinazioni di numeri in un intervallo storico di estrazioni della lotteria

# Il codice analizza le estrazioni della lotteria per determinare la frequenza di presenza di combinazioni di numeri in un intervallo storico, 
# utilizzando una base di calcolo rappresentata da una specifica estrazione.
# Il codice è strutturato per esaminare combinazioni di numeri da 1 a 6, calcolando la frequenza media di presenza per ciascuna classe di combinazioni 
# e fornendo un riassunto dei risultati.
# Il codice utilizza la libreria Lotteria per caricare le estrazioni, numpy per la manipolazione dei dati, itertools per generare le combinazioni 
# e scipy.special.comb per calcolare i coefficienti binomiali.     
#
# Il codice carica un file CSV contenente le estrazioni della lotteria, seleziona una base di calcolo (la prima estrazione) e un 
# intervallo storico (le estrazioni successive alla base).
# Per ogni classe di combinazioni (da 1 a 6 numeri), il codice calcola le combinazioni possibili della base, 
# conta quante volte ciascuna combinazione appare nell'intervallo storico e calcola la frequenza media di presenza per ogni classe di combinazioni. 
# Infine, stampa i risultati riassuntivi per ogni classe di combinazioni, inclusi
# il numero totale di presenze, il numero di combinazioni con almeno una presenza, la massima frequenza di presenza e la frequenza media.
# 
# Nota: il codice utilizza la libreria Lotteria per caricare le estrazioni, numpy per la manipolazione dei dati, 
# itertools per generare le combinazioni e scipy.special.comb per calcolare i coefficienti binomiali.
#
# Il file CSV deve essere formattato correttamente con le estrazioni della lotteria, e il percorso del file deve essere specificato correttamente nella variabile `file_estrazione`.
# 
# Esempio di output:
# base
# [1, 2, 3, 4, 5, 6]
# Combinazioni di classe 1:
# - Presenze: [10, 15, 20, 25, 30, 35]
# - Valori riassuntivi: [135, 6, 35, 22.5]
# Combinazioni di classe 2:
# - Presenze: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75]
# - Valori riassuntivi: [600, 15, 75, 40.0]
# ...   
from lotteria import Lotteria
import numpy as np
from itertools import combinations
from scipy.special import comb
from scipy import stats

def print_dati_grafico(s,i):    
    for j in range(0,4):
        pippo = [x[j] for x in s]
        print(f"\nElenco colonna {j+1} per classe k {i+1}:")
        (uniq, freq) = (np.unique(pippo, return_counts=True))
        print(np.column_stack((uniq,freq)))

file_estrazione = "./data/estrazioni_2026_2006.csv"
ll = Lotteria(file_estrazione)
# Matrice di input
storico_estrazioni = ll.carica_storico(1000)
len_estrazioni = len(storico_estrazioni)

vettore_combinazioni = [comb(6, k, exact=True) for k in range(1,7)] # vettore coefficienti binomiali di 6 su k

size_intervallo = 15
max_basi = len_estrazioni - size_intervallo
        
presenze = [[] for i in range(0,6)]

print("Calcolo presenze per ogni estrazione:")
for e in range(max_basi):
    print("Vettori di input:", e+1, end='\r')
    
    base = storico_estrazioni[e, :].copy()
    matrice_intervallo = storico_estrazioni[e+1:e+1+size_intervallo]  # Prende le righe successive alla prima come matrice di intervallo
    combinazioni_base = [list(combinations(base, k)) for k in range(1,7)] # combinazioni della base

    for i in range(0,6): # per ognuna delle 6 combinazioni su tutte le classi k
        pe = np.empty([0, vettore_combinazioni[i]], dtype=int) # vettore di presenze per ogni combinazione di classe k, inizializzato a zero
        for j in range(0, len(combinazioni_base[i])): # per ogni singola combinazione in classe k
            cb = np.array(combinazioni_base[i][j]) # combinazione della base da esaminare
            ip = 0 # contatore presenze
            for m in range(0, len(matrice_intervallo)): # conta le presenze di una singola combinazione nel subset storico
                if (len(np.setdiff1d(cb, matrice_intervallo[m])) == 0): # se la combinazione e' contenuta nell'estrazione matrice_intervallo[m]
                    ip += 1 
            pe = np.append(pe, ip)
        if sum(pe) > 0 or presenze[i] != [] : # se ci sono presenze, aggiungi il vettore di presenze alla lista delle presenze per la classe k
            presenze[i].append(pe)

# Scarico i risultati riassuntivi per ogni classe k
# print("\n\nProduzione file presenze:")
# for i in range(0,6):
#     #print(f"Valori riassuntivi per classe k {i+1}:", pluto[i])
#     nome_file_out = f"Presenze_{i+1}_{size_intervallo}.csv"   
#     print(nome_file_out)
#     with open(nome_file_out, 'w') as f:
#         for j, presenza in enumerate(presenze[i]):
#             #print(f"Presenza {j+1} per classe k {i+1}: {presenza}")
#             r = ';'.join(str(element) for element in sorted(presenza))
#             s = str(int(sum(presenza))) + ";" + str(int(np.count_nonzero(presenza))) + ";" + str(int(max(presenza))) + ";" + str(np.round(np.mean(presenza),2)) + ";" + str(np.median(presenza)) + ";" + str(stats.mode(presenza).mode)     
#             #print(f"Presenza {j+1} per classe k {i+1}: {r} - Valori riassuntivi: {s}")
#             f.write(r+';'+s+'\n')
#         f.close()

sommari = [[] for i in range(0,6)]
print(f"\n\nProduzione file sommario per {len_estrazioni} estrazioni in intervalli di {size_intervallo} estrazioni:")
for i in range(0,6):
    for j, presenza in enumerate(presenze[i]):
        s = []
        s.append(int(sum(presenza)))
        s.append(int(np.count_nonzero(presenza)))
        s.append(vettore_combinazioni[i] -int(np.count_nonzero(presenza)))
        s.append(int(max(presenza)))
        sommari[i].append(s)

print ("\nAnalisi sommario: (totale presenze, combinazioni con almeno una presenza, combinazioni senza presenze, massima frequenza di presenza  per classe k)") 
for i in range(0,6):     
    # for sommario in sommari[i]:
    #     print(f"Valori riassuntivi per classe k {i+1}: {sommario}")
    s = sommari[i]
    #print_dati_grafico(s, i)
    if len(s) > 0: 
        t = np.sum(s, axis=0)
        u = np.round(np.mean(s, axis=0), 2)
        x = np.round(np.std(s, axis=0), 2) 
        v = np.median(s, axis=0)
        wm, wc = stats.mode(s)[0], stats.mode(s)[1]
        y = np.min(s, axis=0)
        z = np.max(s, axis=0)
        sk = stats.skew(s, axis=0)
        ku = stats.kurtosis(s, axis=0)  
        match i:
            case 0: print(f"\nValori per estratto singolo")
            case 1: print(f"\nValori per coppie di estratti")
            case 2: print(f"\nValori per terne di estratti")
            case 3: print(f"\nValori per quaterne di estratti")
            case 4: print(f"\nValori per cinquine di estratti")
            case 5: print(f"\nValori per sestine di estratti")
        print(f"- sum: {t}, \n- mean: {u}, \n- std: {x}, \n- median: {v}, \n- mode: {wm}, {wc}, \n- min: {y}, \n- max: {z}")
        print(f"- skewness: {np.round(sk, 2)}")
        print(f"- kurtosis: {np.round(ku, 2)}")
        
        print(f"    Totale presenze                      : {np.round(u[0] - x[0], 0)} .. {np.round(u[0] + x[0], 0)}")
        print(f"    Combinazioni con almeno una presenza : {np.round(u[1] - x[1], 0)} .. {np.round(u[1] + x[1], 0)}")
        print(f"    Combinazioni senza presenze          : {np.round(u[2] - x[2], 0)} .. {np.round(u[2] + x[2], 0)}")
        print(f"    Massima frequenza di presenza        : {np.round(u[3] - x[3], 0)} .. {np.round(u[3] + x[3], 0)}")

Calcolo presenze per ogni estrazione:
Vettore di input: 984

Produzione file sommario per 1000 estrazioni in intervalli di 15 estrazioni:

Analisi sommario: (totale presenze, combinazioni con almeno una presenza, combinazioni senza presenze, massima frequenza di presenza  per classe k)

Valori per singolo estratto
- sum: [5851 3817 2093 2248], 
- mean: [5.94 3.88 2.12 2.28], 
- std: [2.34 1.21 1.21 0.88], 
- median: [6. 4. 2. 2.], 
- mode: [5 4 2 2], [160 313 313 465], 
- min: [1 1 0 1], 
- max: [14  6  5  6]
    Totale presenze                     : 3.6 .. 8.28 - 4.0 .. 8.0
    Combinazioni con almeno una presenza: 2.67 .. 5.09 - 3.0 .. 5.0
    Combinazioni senza presenze         : 0.91 .. 3.33 - 1.0 .. 3.0
    Massima frequenza di presenza       : 1.4 .. 3.16 - 1.0 .. 3.0
    Skewness                             : [ 0.35679972 -0.30998219  0.30998219  0.6077296 ]
    Kurtosis                             : [ 0.06354102 -0.35999356 -0.35999356  0.60404763]

Valori per coppie di estratt